# Notebook 02 — Preprocessing & Spatial Aggregation

This notebook:
- Aggregates satellite datasets to monthly resolution
- Reduces pixel-level data to **district-level means**
- Produces clean, policy-ready tables for index construction

All processing is conducted using **Google Earth Engine** and exported for local analysis.


In [1]:
import ee
import geemap
import pandas as pd
from datetime import datetime

ee.Initialize()
print("✅ Earth Engine initialized")

✅ Earth Engine initialized


In [2]:
# Load datasets
# Image collections
ndvi_ic = ee.ImageCollection("MODIS/061/MOD13A3")      # Monthly NDVI
lst_ic  = ee.ImageCollection("MODIS/061/MOD11A2")      # 8-day LST
chirps_ic = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")  # Daily rainfall

# Boundaries
gaul2 = ee.FeatureCollection("FAO/GAUL/2015/level2")
gaul0 = ee.FeatureCollection("FAO/GAUL/2015/level0")

afg_geom = gaul0.filter(
    ee.Filter.eq("ADM0_NAME", "Afghanistan")
).geometry()

afg_districts = gaul2.filterBounds(afg_geom)

print("✅ Datasets loaded")


✅ Datasets loaded


In [3]:
# define analysis period
START_DATE = "2019-01-01"
END_DATE   = "2024-12-31"

print(f"Analysis period: {START_DATE} to {END_DATE}")


Analysis period: 2019-01-01 to 2024-12-31


In [4]:
# helper function : monthly date list 
def month_range(start, end):
    dates = pd.date_range(start=start, end=end, freq="MS")
    return [(d.strftime("%Y-%m-%d"),
             (d + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d"))
            for d in dates]

months = month_range(START_DATE, END_DATE)
print(f"Total months: {len(months)}")


Total months: 72


NDVI Monthly Preprocessing - MODIS NDVI is already monthly we just scale it and clip it

In [14]:
def get_monthly_ndvi(start, end):
    ic = ndvi_ic.filterDate(start, end)
    
    img = ee.Image(
        ee.Algorithms.If(
            ic.size().gt(0),
            ic.mean(),
            ee.Image.constant(0).rename("NDVI")
        )
    )
    
    img = (
        img
        .select("NDVI")
        .multiply(0.0001)
        .clip(afg_geom)
    )
    
    return img.set("system:time_start", ee.Date(start).millis())


LST monthly aggregation - MOD11A2 is 8-day, so we average to monthly and convert units.

In [15]:
def get_monthly_lst(start, end):
    img = (
        lst_ic
        .filterDate(start, end)
        .select("LST_Day_1km")
        .mean()
        .multiply(0.02)    # scale factor
        .subtract(273.15) # Kelvin → Celsius
        .clip(afg_geom)
    )
    return img.set("system:time_start", ee.Date(start).millis())


Rainfall monthly aggregation - CHIRPS daily → monthly sum.

In [16]:
def get_monthly_rainfall(start, end):
    img = (
        chirps_ic
        .filterDate(start, end)
        .select("precipitation")
        .sum()
        .clip(afg_geom)
    )
    return img.set("system:time_start", ee.Date(start).millis())


Build FeatureCollections

In [17]:
# Reduce each month to district means
def reduce_to_districts(img, band_name, date_str):
    reduced = img.reduceRegions(
        collection=afg_districts,
        reducer=ee.Reducer.mean(),
        scale=1000
    )
    
    return reduced.map(
        lambda f: f.set({
            "date": date_str,
            "variable": band_name
        })
    )


In [18]:
# Loop through months & build FeatureCollections
all_features = []

for start, end in months:
    print(f"Processing {start[:7]}")

    ndvi_img = get_monthly_ndvi(start, end)
    lst_img  = get_monthly_lst(start, end)
    rain_img = get_monthly_rainfall(start, end)

    all_features.append(
        reduce_to_districts(ndvi_img, "NDVI", start)
    )
    all_features.append(
        reduce_to_districts(lst_img, "LST_C", start)
    )
    all_features.append(
        reduce_to_districts(rain_img, "RAIN_MM", start)
    )

final_fc = ee.FeatureCollection(all_features).flatten()

print("✅ Monthly district aggregation complete")


Processing 2019-01
Processing 2019-02
Processing 2019-03
Processing 2019-04
Processing 2019-05
Processing 2019-06
Processing 2019-07
Processing 2019-08
Processing 2019-09
Processing 2019-10
Processing 2019-11
Processing 2019-12
Processing 2020-01
Processing 2020-02
Processing 2020-03
Processing 2020-04
Processing 2020-05
Processing 2020-06
Processing 2020-07
Processing 2020-08
Processing 2020-09
Processing 2020-10
Processing 2020-11
Processing 2020-12
Processing 2021-01
Processing 2021-02
Processing 2021-03
Processing 2021-04
Processing 2021-05
Processing 2021-06
Processing 2021-07
Processing 2021-08
Processing 2021-09
Processing 2021-10
Processing 2021-11
Processing 2021-12
Processing 2022-01
Processing 2022-02
Processing 2022-03
Processing 2022-04
Processing 2022-05
Processing 2022-06
Processing 2022-07
Processing 2022-08
Processing 2022-09
Processing 2022-10
Processing 2022-11
Processing 2022-12
Processing 2023-01
Processing 2023-02
Processing 2023-03
Processing 2023-04
Processing 2

In [19]:
# Export to CSV
export_task = ee.batch.Export.table.toDrive(
    collection=final_fc,
    description="afg_monthly_district_climate",
    fileFormat="CSV"
)

export_task.start()
print("📤 Export started — check Google Drive")

📤 Export started — check Google Drive


In [26]:
for t in ee.batch.Task.list():
    print(t.status())


{'state': 'COMPLETED', 'description': 'afg_monthly_district_climate', 'priority': 100, 'creation_timestamp_ms': 1767240935910, 'update_timestamp_ms': 1767242574592, 'start_timestamp_ms': 1767240941337, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/'], 'attempt': 1, 'batch_eecu_usage_seconds': 6221.060546875, 'id': '4OQIKUMUUSNH3BS73AES647X', 'name': 'projects/ee-af-air-quality/operations/4OQIKUMUUSNH3BS73AES647X'}
{'state': 'FAILED', 'description': 'afg_monthly_district_climate', 'priority': 100, 'creation_timestamp_ms': 1767240557588, 'update_timestamp_ms': 1767240570028, 'start_timestamp_ms': 1767240566537, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'batch_eecu_usage_seconds': 24.63850212, 'error_message': "Image.clip: Can't transform (0.0,0.0)", 'id': 'GOHD7QQV2OINLBER43USHERA', 'name': 'projects/ee-af-air-quality/operations/GOHD7QQV2OINLBER43USHERA'}
